# Requests vs. Playwright: Book Club Event Data Demo

This notebook compares two approaches for fetching book club events:
- **requests** + SerpAPI (scrapes Google Events, returns structured JSON)
- **playwright** (scrapes event pages directly: libraries, bookstores, city calendars)

---

## Part 1: `requests` + SerpAPI

**Run the cell below** to fetch book club events via SerpAPI's Google Events engine (uses `data/scripts/get_seattle_bookclubs_data.py`).

In [1]:
import os
import sys

# Add project root so we can import from data/scripts
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

import pandas as pd

if not os.getenv("api_key"):
    print("[SKIP] No api_key in .env. Set api_key for SerpAPI to run this demo.")
    df_requests = pd.DataFrame(columns=["source", "title", "link", "when", "address"])
    request_count = 0
    n_requests = 0
else:
    try:
        from data.scripts.get_seattle_bookclubs_data import fetch_events
    except ImportError:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "get_seattle_bookclubs_data",
            os.path.join(PROJECT_ROOT, "data", "scripts", "get_seattle_bookclubs_data.py")
        )
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fetch_events = mod.fetch_events

    df_requests, request_count = fetch_events(max_requests=3)
    df_requests = df_requests.assign(source="SerpAPI (Google Events)")
    n_requests = len(df_requests)

print(f"\n{'='*50}")
print(f"  requests + SerpAPI: {n_requests} events scraped ({request_count} API calls)")
print(f"{'='*50}\n")
if n_requests > 0:
    display(df_requests[["source", "title", "link", "when", "address"]].head(10))


  requests + SerpAPI: 30 events scraped (3 API calls)



,source,title,link,when,address
0,SerpAPI (Google Events),Tool Library Book Club,https://seattlereconomy.org/event/tool-library...,"Mon, Mar 16, 7:00 – 8:30 PM","[NE Seattle Tool Library, 10228 Fischer Pl NE,..."
1,SerpAPI (Google Events),Seattle Book Club Anniversary Party,https://www.eventbrite.com/e/seattle-book-club...,"Sat, Mar 21, 5 – 8 PM","[Old Stove Brewing Ship Canal, 600 W Nickerson..."
2,SerpAPI (Google Events),"Blush & Lust Book Club | And Now, Back To You",https://www.eventbrite.com/e/blush-lust-book-c...,"Tue, Mar 24, 6:30 – 8:00 PM","[Lovestruck in Seattle, 8507 35th Ave NE, Seat..."
3,SerpAPI (Google Events),EnGAYging with Sapphistry Book Club,https://www.eventbrite.com/e/engayging-with-sa...,"Sun, Mar 15, 2 – 3 PM","[Redmond Library, 15990 NE 85th St, Redmond, WA]"
4,SerpAPI (Google Events),Book Club,https://issaquahhighlands.com/event/book-club-...,"Tue, Mar 10, 7:00 – 8:30 PM","[Blakely Hall, 2550 NE Park Dr #5, Issaquah, WA]"
5,SerpAPI (Google Events),A Year of Emily Henry Book Club | People We Me...,https://www.eventbrite.com/e/a-year-of-emily-h...,"Wed, Mar 18, 6:30 – 8:00 PM","[Lovestruck in Seattle, 8507 35th Ave NE, Seat..."
6,SerpAPI (Google Events),Silent Book Club,https://populusseattle.com/event/silent-book-c...,"Thu, Apr 16, 6 – 8 PM","[Populus Seattle, 100 S King St, Seattle, WA]"
7,SerpAPI (Google Events),Maple Leaf Silent Book Club,https://do206.com/events/2026/3/15/maple-leaf-...,"Sun, Mar 15, 3 – 5 PM","[Watershed Pub & Kitchen, 10104 3rd Ave NE, Se..."
8,SerpAPI (Google Events),Chapter One Book Club,https://kcls.bibliocommons.com/events/691e614b...,"Tue, Mar 10, 4 – 5 PM","[Renton Highlands Library, 2801 NE 10th St, Re..."
9,SerpAPI (Google Events),March Book Club!,https://www.eventbrite.com/e/march-book-club-t...,"Fri, Mar 27, 7 – 8 PM","[Sfingiday, 601 N 35th St, Seattle, WA]"


Note: This is only a snippet of the events scraped. We capped it to save tokens, but we get roughly 100 events.

---

## Part 2: `playwright` 

**Run the cell below** to scrape book club events from libraries, bookstores, and city calendars.

**Sources:** Secret Garden Books, Third Place Books, Seattle Public Library, Seattle.gov, Elliott Bay Book Company

*(Requires: `pip install playwright` and `playwright install chromium`)*

In [2]:
import pandas as pd
try:
    from playwright.async_api import async_playwright
except ImportError:
    print("[SKIP] playwright not installed. Run: pip install playwright && playwright install chromium")
    df_playwright = pd.DataFrame()
    n_playwright = 0
else:
    async def scrape_secret_garden(page):
        events = []
        try:
            await page.goto("https://secretgardenbooks.com/events", wait_until="networkidle", timeout=15000)
            await page.wait_for_selector("a[href*='/event/']", timeout=5000)
            links = await page.query_selector_all('a[href*="/event/"]')
            seen = set()
            for link in links:
                href = (await link.get_attribute("href")) or ""
                if "/event/" not in href or href in seen:
                    continue
                seen.add(href)
                title = (await link.inner_text()).strip()
                if not title or len(title) < 4:
                    continue
                full_url = href if href.startswith("http") else f"https://secretgardenbooks.com{href}"
                events.append({"source": "Secret Garden Books", "title": title[:200], "link": full_url, "when": "", "address": "2214 NW Market St, Seattle, WA"})
        except Exception as e:
            print(f"[Secret Garden] Error: {e}")
        return events

    async def scrape_third_place(page):
        events = []
        try:
            await page.goto("https://www.thirdplacebooks.com/events", wait_until="networkidle", timeout=15000)
            await page.wait_for_selector("a[href*='/event/']", timeout=8000)
            links = await page.query_selector_all("a[href*='/event/']")
            seen = set()
            for link in links:
                href = (await link.get_attribute("href")) or ""
                if "/event/" not in href or href in seen:
                    continue
                seen.add(href)
                title = (await link.inner_text()).strip()
                if not title or len(title) < 5:
                    continue
                full_url = href if href.startswith("http") else f"https://www.thirdplacebooks.com{href}"
                events.append({"source": "Third Place Books", "title": title[:200], "link": full_url, "when": "", "address": ""})
        except Exception as e:
            print(f"[Third Place] Error: {e}")
        return events

    async def scrape_spl(page):
        events = []
        try:
            await page.goto("https://www.spl.org/event-calendar", wait_until="networkidle", timeout=20000)
            await page.wait_for_load_state("networkidle")
            await page.wait_for_timeout(2000)
            links = await page.query_selector_all('a[href*="/event"], a[href*="event-calendar"], .event-title a, .event a')
            seen = set()
            for link in links:
                href = (await link.get_attribute("href")) or ""
                if not href or href in seen:
                    continue
                seen.add(href)
                title = (await link.inner_text()).strip()
                if not title or len(title) < 4:
                    continue
                full_url = href if href.startswith("http") else f"https://www.spl.org{href}"
                events.append({"source": "Seattle Public Library", "title": title[:200], "link": full_url, "when": "", "address": ""})
        except Exception as e:
            print(f"[SPL] Error: {e}")
        return events

    async def scrape_seattle_gov(page):
        events = []
        try:
            await page.goto("https://www.seattle.gov/event-calendar", wait_until="networkidle", timeout=15000)
            await page.wait_for_load_state("networkidle")
            await page.wait_for_timeout(1500)
            links = await page.query_selector_all(
                'a[href*="/event"], a[href*="event-calendar"], .event a, .event-title a, [class*="event"] a'
            )
            seen = set()
            for link in links:
                href = (await link.get_attribute("href")) or ""
                if not href or href in seen:
                    continue
                if "event" not in href.lower() and "calendar" not in href.lower():
                    continue
                seen.add(href)
                title = (await link.inner_text()).strip()
                if not title or len(title) < 4:
                    continue
                full_url = href if href.startswith("http") else f"https://www.seattle.gov{href}"
                events.append({"source": "Seattle.gov", "title": title[:200], "link": full_url, "when": "", "address": ""})
        except Exception as e:
            print(f"[Seattle.gov] Error: {e}")
        return events

    async def scrape_elliott_bay(page):
        events = []
        try:
            await page.goto("https://www.elliottbaybook.com/events/current/calendar", wait_until="networkidle", timeout=20000)
            await page.wait_for_load_state("networkidle")
            await page.wait_for_timeout(3000)
            links = await page.query_selector_all('a[href*="/event"], a[href*="events"]')
            seen = set()
            for link in links:
                href = (await link.get_attribute("href")) or ""
                if not href or href in seen:
                    continue
                if "event" not in href.lower():
                    continue
                seen.add(href)
                title = (await link.inner_text()).strip()
                if not title or len(title) < 4:
                    continue
                full_url = href if href.startswith("http") else f"https://www.elliottbaybook.com{href}"
                events.append({"source": "Elliott Bay Book Company", "title": title[:200], "link": full_url, "when": "", "address": "1521 10th Ave, Seattle, WA"})
        except Exception as e:
            print(f"[Elliott Bay] Error: {e}")
        return events

    async def run_playwright():
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            context = await browser.new_context(
                user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
            )
            page = await context.new_page()
            page.set_default_timeout(15000)
            all_events = []
            scrapers = [
                ("Secret Garden Books", scrape_secret_garden),
                ("Third Place Books", scrape_third_place),
                ("Seattle Public Library", scrape_spl),
                ("Seattle.gov", scrape_seattle_gov),
                ("Elliott Bay Book Company", scrape_elliott_bay),
            ]
            for name, scraper in scrapers:
                print(f"Scraping {name}...")
                events = await scraper(page)
                all_events.extend(events)
                print(f"  -> {len(events)} events")
            await browser.close()
        return all_events

    all_events = await run_playwright()
    df_playwright = pd.DataFrame(all_events)
    n_playwright = len(df_playwright)
    print(f"\n{'='*50}")
    print(f"  playwright: {n_playwright} events scraped (5 sources)")
    print(f"{'='*50}\n")
    display(df_playwright[["source", "title", "link", "when", "address"]].head(10))

Scraping Secret Garden Books...
  -> 1 events
Scraping Third Place Books...
  -> 6 events
Scraping Seattle Public Library...
  -> 3 events
Scraping Seattle.gov...
  -> 12 events
Scraping Elliott Bay Book Company...
  -> 0 events

  playwright: 22 events scraped (5 sources)



,source,title,link,when,address
0,Secret Garden Books,MARCH BOOK CLUB: I WHO HAVE NEVER KNOWN MEN,https://secretgardenbooks.com/event/2026-03-26...,,"2214 NW Market St, Seattle, WA"
1,Third Place Books,Meg Richman presents 'Freya the Deer',https://www.thirdplacebooks.com/event/meg-richman,,
2,Third Place Books,"Anjali Enjeti, Tamiko Nimura, and Simona Supek...",https://www.thirdplacebooks.com/event/enjeti-n...,,
3,Third Place Books,African-American Writers' Alliance Open Mic Night,https://www.thirdplacebooks.com/event/african-...,,
4,Third Place Books,Kathryn Gillespie presents 'The Sound of Feath...,https://www.thirdplacebooks.com/event/kathryn-...,,
5,Third Place Books,Storytime with Southeast Youth & Family Services,https://www.thirdplacebooks.com/event/storytim...,,
6,Third Place Books,Town Hall Seattle: Matthew Sutton with Bill Ra...,https://www.thirdplacebooks.com/event/town-hal...,,
7,Seattle Public Library,Skip to Main Content,https://www.spl.org/event-calendar#main-content,,
8,Seattle Public Library,Event Calendar,https://www.spl.org/event-calendar,,
9,Seattle Public Library,BACK TO\nTOP,https://www.spl.org/event-calendar#site-container,,


---

## Comparison


In [3]:
# Event count comparison (run after both cells above)
print("\n" + "="*50)
print("  EVENT COUNT COMPARISON")
print("="*50)
print(f"  requests + SerpAPI:  {n_requests} events")
print(f"  playwright:          {n_playwright} events")
print("="*50)


  EVENT COUNT COMPARISON
  requests + SerpAPI:  30 events
  playwright:          22 events
